# Analyse Exploratoire v2 — Modele CLV Nouvelles Acquisitions

Version 2 structuree autour des **4 questions metier**, avec ventilation **exhaustive**
selon 6 caracteristiques client (cohorte / canal / tranche de revenu / residence / age / CSP).

> Donnees propres (`clean_data/*.parquet`) produites par `data_preparation.ipynb`
> (inclut desormais **canal** sur STOCK et **clean_upstream** BDL/BPL).

## Plan
0. **Fondations** — chargement, panel famille, tenure, moteur de ventilation
1. **Qui sont nos nouveaux clients ?**
2. **Activite des premiers mois** — churn, upstream, equipement & encours M+3/6/12, trajectoires
3. **Quel revenu genèrent-ils ?** — PnB M+3/6/12, PnB Y1→Y5, concentration, par produit
4. **Comment calibrer le modèle ?** — equipement/encours/PnB M+3 vs M+12 vs Y5, marges, deciles

> Code en anglais, narration en francais. Exercice refait **hors mineurs / hors etudiants** via `EXCLUDE_*`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

CLEAN_DIR = Path("../clean_data")
FIG_DIR = Path("../eda_figures_v2")
FIG_DIR.mkdir(exist_ok=True)

EPOCH_YEAR = 1994
def idx_to_year(idx): return EPOCH_YEAR + (int(idx) - 1) // 12
def idx_to_ym(idx):   return (EPOCH_YEAR + (int(idx) - 1) // 12, (int(idx) - 1) % 12 + 1)

NBI_COL = "revenus_eur_avec_liquidity"

FAM_ORDER  = ["epargne", "invest", "credit_immo", "autres_cr", "cartes"]
ENC_FAMS   = ["epargne", "invest", "credit_immo", "autres_cr"]   # cartes = pas d'encours
FAM_COLORS = {"epargne": "#06B6D4", "invest": "#7C3AED", "credit_immo": "#EC4899",
              "autres_cr": "#F59E0B", "cartes": "#84CC16"}
FAM_LABELS = {"epargne": "Epargne", "invest": "Investissements", "credit_immo": "Credit immo",
              "autres_cr": "Autres credits", "cartes": "Autre PNB"}
FAMILY_ENCOURS = {
    "epargne":      ("savings",   ["avoirs_eur"]),
    "invest":       ("invest",    ["cp_depots_a_terme", "cp_depots_en_cc_epar_liq",
                                   "cp_depots_en_cc_placement", "cp_depots_titres", "cp_epargne_a_vue"]),
    "invest_banca": ("banca",     ["capitaux_productifs_eur"]),
    "credit_immo":  ("pret_immo", ["credits_utilises_eur"]),
    "autres_cr":    ("credit",    ["credits_utilises_eur"]),
}

plt.rcParams.update({"figure.figsize": (13, 6), "axes.grid": True, "grid.alpha": 0.25,
                     "axes.titlesize": 12, "axes.labelsize": 10, "figure.dpi": 100})
sns.set_palette("muted")
PURPLE, CYAN, PINK, AMBER = "#7C3AED", "#06B6D4", "#EC4899", "#F59E0B"

def save_fig(name, fig=None):
    fig = fig or plt.gcf()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=140, bbox_inches="tight", facecolor="white")
    plt.show()

print("Setup OK — figures ->", FIG_DIR.resolve())

## 0. Fondations

Chargement, panel famille, axe d'anciennete (`tenure`), et le **moteur de ventilation** :
`break_by`, `break_by_multi`, `curve_by`, `count_by` tracent n'importe quelle metrique client
selon les 6 caracteristiques. Toutes les vues "selon les differentes caracteristiques" en decoulent.

In [ ]:
def load(name):
    p = CLEAN_DIR / f"clean_{name}.parquet"
    if not p.exists():
        print(f"  MANQUANT: {p.name}"); return None
    df = pd.read_parquet(p)
    print(f"  {name:<12}: {len(df):>10,} lignes x {df.shape[1]} cols")
    return df

print("Chargement:")
stock     = load("stock");   flux    = load("flux")
savings   = load("savings"); invest  = load("invest");  banca = load("banca")
cartes    = load("cartes");  credit  = load("credit");  pret_immo = load("pret_immo")
upstream  = load("upstream")     # scope BDL/BPL (id, periode, scope) — peut etre None

PRODUCTS = {"savings": savings, "invest": invest, "banca": banca,
            "cartes": cartes, "credit": credit, "pret_immo": pret_immo}
HAS_CANAL    = stock is not None and "canal" in stock.columns
HAS_UPSTREAM = upstream is not None and len(upstream) > 0
print(f"\ncanal dispo: {HAS_CANAL} | upstream dispo: {HAS_UPSTREAM}")

In [ ]:
# Periode d'acquisition, tenure, alias NBI
acq = stock.groupby("identifiant_tiers")["periode"].min().rename("acq_period")

def add_tenure(df):
    if df is None: return None
    df = df.merge(acq, on="identifiant_tiers", how="left")
    df["tenure"] = (df["periode"] - df["acq_period"]).astype("Int64")
    return df

stock = add_tenure(stock); flux = add_tenure(flux)
for k in list(PRODUCTS): PRODUCTS[k] = add_tenure(PRODUCTS[k])
savings, invest, banca, cartes, credit, pret_immo = (PRODUCTS[k] for k in
    ["savings", "invest", "banca", "cartes", "credit", "pret_immo"])
if NBI_COL in flux.columns: flux["nbi"] = flux[NBI_COL]

max_tenure = flux.groupby("identifiant_tiers")["tenure"].max()
cohort_of  = acq.map(idx_to_year)
print(f"Clients: {stock['identifiant_tiers'].nunique():,} | anciennete mediane {int(max_tenure.median())} mois")

In [ ]:
# ── Panel famille : encours + NBI par (client, periode, famille) ─────────────
def agg_encours(df, cols):
    if df is None: return None
    cols = [c for c in cols if c in df.columns]
    if not cols: return None
    tmp = df[["identifiant_tiers", "periode"]].copy()
    tmp["encours"] = df[cols].sum(axis=1, min_count=1)
    return tmp.groupby(["identifiant_tiers", "periode"], as_index=False)["encours"].sum()

def agg_nbi(df):
    if df is None or NBI_COL not in df.columns: return None
    tmp = df[["identifiant_tiers", "periode", NBI_COL]].rename(columns={NBI_COL: "nbi"})
    return tmp.groupby(["identifiant_tiers", "periode"], as_index=False)["nbi"].sum()

def combine(a, b, val):
    parts = [x for x in (a, b) if x is not None]
    if not parts: return None
    return pd.concat(parts).groupby(["identifiant_tiers", "periode"], as_index=False)[val].sum()

enc = {"epargne": agg_encours(savings, FAMILY_ENCOURS["epargne"][1]),
       "invest": combine(agg_encours(invest, FAMILY_ENCOURS["invest"][1]),
                         agg_encours(banca, FAMILY_ENCOURS["invest_banca"][1]), "encours"),
       "credit_immo": agg_encours(pret_immo, FAMILY_ENCOURS["credit_immo"][1]),
       "autres_cr": agg_encours(credit, FAMILY_ENCOURS["autres_cr"][1])}
nbi = {"epargne": agg_nbi(savings),
       "invest": combine(agg_nbi(invest), agg_nbi(banca), "nbi"),
       "credit_immo": agg_nbi(pret_immo), "autres_cr": agg_nbi(credit), "cartes": agg_nbi(cartes)}

parts = []
for fam in FAM_ORDER:
    n, e = nbi.get(fam), enc.get(fam)
    if n is None and e is None: continue
    base = n.copy() if n is not None else e[["identifiant_tiers", "periode"]].copy()
    if n is None: base["nbi"] = np.nan
    base = base.merge(e, on=["identifiant_tiers", "periode"], how="outer") if e is not None else base.assign(encours=np.nan)
    base["family"] = fam
    parts.append(base)
panel = pd.concat(parts, ignore_index=True).merge(acq, on="identifiant_tiers", how="left").dropna(subset=["acq_period"])
panel["tenure"] = (panel["periode"] - panel["acq_period"]).astype(int)
panel["cohort"] = panel["acq_period"].map(idx_to_year)
panel = panel[panel["tenure"] >= 0]

def eligible_clients(h): return set(max_tenure[max_tenure >= h - 1].index)
def value_at_tenure(values, fam, t):
    p = panel[(panel["family"] == fam) & (panel["tenure"] == t)]
    return p.groupby("identifiant_tiers")[values].sum()
def cum_nbi_client(h, fam=None):
    p = panel if fam is None else panel[panel["family"] == fam]
    return p[p["tenure"] < h].groupby("identifiant_tiers")["nbi"].sum()
def nbi_year(y):
    lo, hi = (y - 1) * 12, y * 12
    return panel[(panel["tenure"] >= lo) & (panel["tenure"] < hi)].groupby("identifiant_tiers")["nbi"].sum()
print(f"Panel: {len(panel):,} lignes")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# MOTEUR DE VENTILATION — 6 caracteristiques client
# ══════════════════════════════════════════════════════════════════════════
latest = stock.sort_values("periode").groupby("identifiant_tiers").last()

REV_ORDER = ["Moins de 35.999 EUR", "Entre 36.000 et 53.999 EUR",
             "Entre 54.000 et 119.999 EUR", "Plus de 120.000 EUR"]
AGE_LABELS = ["<18", "18-24", "25-34", "35-44", "45-54", "55-64", "65+"]

cli = pd.DataFrame(index=latest.index)
cli["cohorte"] = cohort_of.reindex(cli.index)
cli["canal"]   = latest["canal"] if "canal" in latest.columns else "Inconnu"
rev = latest["revenu_total"].astype(str).str.strip() if "revenu_total" in latest.columns else pd.Series("Inconnu", index=cli.index)
rev = rev.where(rev.isin(REV_ORDER), "Inconnu")
cli["tranche_revenu"] = pd.Categorical(rev, categories=REV_ORDER + ["Inconnu"], ordered=True)
cli["residence"] = (latest["frontalier_o_n"].astype(str).str.strip().str.upper()
                    .map({"O": "Frontalier", "N": "Resident"}).fillna("Inconnu")
                    if "frontalier_o_n" in latest.columns else "Inconnu")
age = pd.to_numeric(latest["age"], errors="coerce") if "age" in latest.columns else pd.Series(np.nan, index=cli.index)
cli["age_num"] = age
cli["tranche_age"] = pd.cut(age, [-1, 17, 24, 34, 44, 54, 64, 200], labels=AGE_LABELS)
cli["csp"] = (latest["segment_ameliore"].astype(str).str.strip()
              if "segment_ameliore" in latest.columns else "Inconnu")

# ── Filtres exercice : hors mineurs / hors etudiants ────────────────────────
EXCLUDE_MINORS   = True
EXCLUDE_STUDENTS = False    # -> True si une modalite etudiante existe dans la CSP
student_mask = cli["csp"].str.contains("etud|student", case=False, na=False)
keep = pd.Series(True, index=cli.index)
if EXCLUDE_MINORS:   keep &= (cli["age_num"] >= 18) | cli["age_num"].isna()
if EXCLUDE_STUDENTS: keep &= ~student_mask
cli_all = cli.copy()
cli = cli[keep]

CHARACS = ["cohorte", "canal", "tranche_revenu", "residence", "tranche_age", "csp"]
CHARAC_LABELS = {"cohorte": "Cohorte d'acquisition", "canal": "Canal (EER)",
                 "tranche_revenu": "Tranche de revenu", "residence": "Residence",
                 "tranche_age": "Tranche d'age", "csp": "CSP (segment ameliore)"}
CSP_TOP = 8

def _cats(ch):
    vc = cli[ch].value_counts()
    if ch in ("tranche_revenu", "tranche_age", "cohorte"): return [c for c in vc.sort_index().index]
    if ch == "csp": return [c for c in vc.head(CSP_TOP).index]
    return [c for c in vc.sort_values(ascending=False).index]
def _order(ch, s): return s.reindex(_cats(ch)).dropna()

print(f"Clients: {len(cli_all):,} -> {len(cli):,} apres filtres "
      f"(mineurs exclus={EXCLUDE_MINORS}, etudiants exclus={EXCLUDE_STUDENTS})")
print("Modalites CSP (top):", cli['csp'].value_counts().head(CSP_TOP).to_dict())

In [ ]:
# ── Fonctions de trace : count_by / break_by / break_by_multi / curve_by ────
def count_by(base=None, title="Nombre d'acquisitions par caracteristique", fig_name=None):
    b = cli if base is None else base
    fig, axes = plt.subplots(2, 3, figsize=(19, 10))
    for ax, ch in zip(axes.ravel(), CHARACS):
        vc = _order(ch, b[ch].value_counts())
        vc.plot(kind="bar", ax=ax, color=PURPLE, edgecolor="white")
        ax.set_title(CHARAC_LABELS[ch]); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
        for i, v in enumerate(vc.values): ax.text(i, v, f"{int(v):,}", ha="center", va="bottom", fontsize=8)
    fig.suptitle(f"{title}  (n={len(b):,} clients)", fontsize=15); plt.tight_layout()
    save_fig(fig_name) if fig_name else plt.show()

def break_by(value, agg="mean", title="", pct=False, ylabel="", fig_name=None):
    """value: Series indexee par client -> agg(value) par caracteristique (grille 2x3)."""
    val = value.reindex(cli.index)
    fig, axes = plt.subplots(2, 3, figsize=(19, 10))
    for ax, ch in zip(axes.ravel(), CHARACS):
        d = pd.DataFrame({"v": val, "g": cli[ch]}).dropna(subset=["g"])
        g = _order(ch, d.groupby("g", observed=True)["v"].agg(agg)).dropna()
        (g * 100 if pct else g).plot(kind="bar", ax=ax, color=CYAN, edgecolor="white")
        ax.set_title(CHARAC_LABELS[ch]); ax.set_xlabel(""); ax.set_ylabel(ylabel); ax.tick_params(axis="x", rotation=30)
    fig.suptitle(title, fontsize=15); plt.tight_layout()
    save_fig(fig_name) if fig_name else plt.show()

def break_by_multi(series_map, title="", pct=False, ylabel="", fig_name=None):
    """series_map: {label: Series} -> barres groupees (une par label) par caracteristique."""
    cols = pd.DataFrame({lab: s.reindex(cli.index) for lab, s in series_map.items()})
    fig, axes = plt.subplots(2, 3, figsize=(19, 10))
    for ax, ch in zip(axes.ravel(), CHARACS):
        d = cols.copy(); d["g"] = cli[ch]
        g = d.dropna(subset=["g"]).groupby("g", observed=True).mean(numeric_only=True).reindex(_cats(ch)).dropna(how="all")
        (g * 100 if pct else g).plot(kind="bar", ax=ax, edgecolor="white")
        ax.set_title(CHARAC_LABELS[ch]); ax.set_xlabel(""); ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", rotation=30); ax.legend(fontsize=7)
    fig.suptitle(title, fontsize=15); plt.tight_layout()
    save_fig(fig_name) if fig_name else plt.show()

def curve_by(flag_at, horizons, title, fig_name=None):
    """flag_at(X) -> Series 0/1 (NaN si non eligible). Trace le taux cumule par caracteristique."""
    flags = {X: flag_at(X).reindex(cli.index) for X in horizons}
    fig, axes = plt.subplots(2, 3, figsize=(19, 10))
    for ax, ch in zip(axes.ravel(), CHARACS):
        for cat in _cats(ch):
            idx = cli.index[cli[ch] == cat]
            ys = [flags[X].reindex(idx).mean() * 100 for X in horizons]
            ax.plot(horizons, ys, "o-", label=str(cat), lw=1.4, ms=3)
        ax.set_title(CHARAC_LABELS[ch]); ax.set_xlabel("Mois (tenure)"); ax.set_ylabel("%"); ax.legend(fontsize=6)
    fig.suptitle(title, fontsize=15); plt.tight_layout()
    save_fig(fig_name) if fig_name else plt.show()

print("Moteur pret : count_by / break_by / break_by_multi / curve_by")

## 1. Qui sont nos nouveaux clients ?

Distribution des acquisitions selon les 6 caracteristiques : **annee (cohorte), canal
(EER digital vs physique), tranche de revenu, residents/frontaliers, age, CSP**.

In [ ]:
count_by(title="Nombre d'acquisitions par caracteristique", fig_name="s1_acquisitions")
print("Repartition des acquisitions :")
for ch in CHARACS:
    vc = _order(ch, cli[ch].value_counts())
    print(f"\n{CHARAC_LABELS[ch]} :")
    for k, v in vc.items():
        print(f"  {str(k):<32} {int(v):>8,}  ({v/len(cli)*100:4.1f}%)")

## 2. Activite des clients sur leurs premiers mois

**Churn** = mois ou le client **disparait de la timeline** (plus d'info), sauf s'il a fait un
**upstream** (passage banque de detail BDL → banque privee BPL, confirme par `clean_upstream`).
On regarde la survie, l'equipement/encours des churners avant depart, la montee en banque privee,
puis les taux d'equipement et montants d'encours a M+3/M+6/M+12 par produit.

In [ ]:
# ── Fondations churn / upstream ─────────────────────────────────────────────
lo_flux  = flux.groupby("identifiant_tiers")["periode"].max()
lo_stock = stock.groupby("identifiant_tiers")["periode"].max()
last_obs = pd.concat([lo_flux, lo_stock], axis=1).max(axis=1)          # dernier mois avec info
DATA_MAX = int(pd.concat([flux["periode"], stock["periode"]]).max())
potential_window = (DATA_MAX - acq)                                    # tenure max observable

# Upstream : 1er passage BPL
if HAS_UPSTREAM:
    up_first = upstream[upstream["scope"] == "BPL"].groupby("identifiant_tiers")["periode"].min()
else:
    up_first = pd.Series(dtype="float64")
upstreamed = pd.Series(cli_all.index.isin(up_first.index), index=cli_all.index)
up_tenure  = (up_first.reindex(cli_all.index) - acq.reindex(cli_all.index))

# Churn : disparait avant la fin des donnees ET pas explique par upstream
disappeared = (last_obs.reindex(cli_all.index) < DATA_MAX)
churned = disappeared & ~upstreamed
churn_tenure = (last_obs.reindex(cli_all.index) + 1 - acq.reindex(cli_all.index)).where(churned)

status = pd.Series("Actif", index=cli_all.index)
status[churned] = "Churn"
status[upstreamed] = "Upstream"           # upstream prime

def churn_flag_at(X):
    elig = potential_window.reindex(cli.index) >= X
    return (churn_tenure.reindex(cli.index) <= X).astype(float).where(elig)
def upstream_flag_at(X):
    elig = potential_window.reindex(cli.index) >= X
    return (up_tenure.reindex(cli.index) <= X).astype(float).where(elig)
def elig_mask(t): return potential_window.reindex(cli.index) >= t
def masked(series, t): return series.reindex(cli.index).where(elig_mask(t))

print("Statut final des clients (fenetre observee) :")
print(status.reindex(cli.index).value_counts().to_string())
print(f"\nDATA_MAX (dernier mois de donnees) = {DATA_MAX} ({idx_to_ym(DATA_MAX)[0]}-{idx_to_ym(DATA_MAX)[1]:02d})")

In [ ]:
# 2.1 — Courbe de churn cumule selon les caracteristiques
HORIZ = [3, 6, 12, 18, 24, 36, 48, 60]
recap = [{"M+": X, "churn_%": round(churn_flag_at(X).mean() * 100, 1),
          "n_eligibles": int(churn_flag_at(X).notna().sum())} for X in HORIZ]
print("Taux de churn cumule (global) :"); print(pd.DataFrame(recap).to_string(index=False))
curve_by(churn_flag_at, HORIZ, "Taux de churn cumule par caracteristique", "s2_churn_curve")
break_by(churn_flag_at(12), pct=True, ylabel="% churn a M+12",
         title="Taux de churn a M+12 par caracteristique", fig_name="s2_churn_m12")

In [ ]:
# 2.2 — Equipement & encours des churners JUSTE AVANT de churner
# (etat au dernier mois observe : tenure = churn_tenure - 1)
own_ten = (churn_tenure - 1).dropna().astype(int).rename("t")     # dernier mois actif
pv = panel.merge(own_ten, left_on="identifiant_tiers", right_index=True)
pv = pv[pv["tenure"] == pv["t"]]
eq_churn = pv[(pv["encours"].fillna(0) > 0) | (pv["nbi"].fillna(0) > 0)].groupby("identifiant_tiers")["family"].nunique()
enc_churn = pv.groupby("identifiant_tiers")["encours"].sum()

churn_idx = status.index[status == "Churn"]
print(f"Churners : {len(churn_idx):,}")
print(f"  Nb produits moyen au dernier mois : {eq_churn.reindex(churn_idx).mean():.2f}")
print(f"  Encours median au dernier mois    : {enc_churn.reindex(churn_idx).median():,.0f} EUR")
# Comparaison equipement churners vs non-churners (a equivalent tenure) : nb produits moyen
break_by(eq_churn.reindex(cli.index), ylabel="Nb produits (dernier mois)",
         title="Equipement des churners avant depart, par caracteristique", fig_name="s2_churn_equip")
break_by(enc_churn.reindex(cli.index), ylabel="Encours (EUR)",
         title="Encours des churners avant depart, par caracteristique", fig_name="s2_churn_encours")

In [ ]:
# 2.3 — Upstream (BDL -> BPL) : courbe + equipement des upstreamers
if HAS_UPSTREAM:
    recap = [{"M+": X, "upstream_%": round(upstream_flag_at(X).mean() * 100, 2),
              "n_eligibles": int(upstream_flag_at(X).notna().sum())} for X in HORIZ]
    print("Taux d'upstream cumule (global) :"); print(pd.DataFrame(recap).to_string(index=False))
    curve_by(upstream_flag_at, HORIZ, "Taux d'upstream (BDL->BPL) cumule par caracteristique", "s2_upstream_curve")
    # Equipement des upstreamers sur leurs premiers mois (nb produits a M+3)
    eq_m3 = panel[(panel["tenure"] == 3) & ((panel["encours"].fillna(0) > 0) | (panel["nbi"].fillna(0) > 0))] \
            .groupby("identifiant_tiers")["family"].nunique()
    up_idx = status.index[status == "Upstream"]
    print(f"\nUpstreamers : {len(up_idx):,} | nb produits moyen a M+3 : {eq_m3.reindex(up_idx).mean():.2f} "
          f"(vs {eq_m3.reindex(cli.index).mean():.2f} tous clients)")
    break_by(upstream_flag_at(24), pct=True, ylabel="% upstream a M+24",
             title="Taux d'upstream a M+24 par caracteristique", fig_name="s2_upstream_m24")
else:
    print("Pas de donnees upstream (clean_upstream absent).")

In [ ]:
# 2.4 — Taux d'equipement a M+3 / M+6 / M+12 par produit (toutes cohortes, presents ou non)
EQ_HOR = [3, 6, 12]
def holds_flag(fam, t):
    held = set(panel[(panel["family"] == fam) & (panel["tenure"] == t) &
                     ((panel["encours"].fillna(0) > 0) | (panel["nbi"].fillna(0) > 0))]["identifiant_tiers"])
    return pd.Series(cli.index.isin(held).astype(float), index=cli.index).where(elig_mask(t))

tbl = []
for fam in FAM_ORDER:
    r = {"famille": FAM_LABELS[fam]}
    for X in EQ_HOR: r[f"equip_M+{X}_%"] = round(holds_flag(fam, X).mean() * 100, 1)
    tbl.append(r)
print("Taux d'equipement par produit :"); print(pd.DataFrame(tbl).to_string(index=False))
# Ventilation exhaustive : un graphe par produit (M+3/6/12 groupes par caracteristique)
for fam in FAM_ORDER:
    break_by_multi({f"M+{X}": holds_flag(fam, X) for X in EQ_HOR}, pct=True, ylabel="% equipes",
                   title=f"Taux d'equipement {FAM_LABELS[fam]} par caracteristique", fig_name=f"s2_equip_{fam}")

In [ ]:
# 2.5 — Montant moyen des encours a M+3 / M+6 / M+12 par produit (detenteurs, par cohorte + caract.)
def enc_at(fam, t):
    s = value_at_tenure("encours", fam, t)
    s = s[s > 0]                                     # detenteurs
    return masked(s, t)

tbl = []
for fam in ENC_FAMS:
    r = {"famille": FAM_LABELS[fam]}
    for X in EQ_HOR:
        v = enc_at(fam, X).dropna()
        r[f"enc_med_M+{X}"] = round(v.median(), 0) if len(v) else np.nan
    tbl.append(r)
print("Encours median des detenteurs par produit :"); print(pd.DataFrame(tbl).to_string(index=False))
for fam in ENC_FAMS:
    break_by_multi({f"M+{X}": enc_at(fam, X) for X in EQ_HOR}, ylabel="Encours moyen (EUR)",
                   title=f"Encours moyen {FAM_LABELS[fam]} (detenteurs) par caracteristique",
                   fig_name=f"s2_enc_{fam}")

In [ ]:
# 2.6 — Trajectoire des encours a partir de l'equipement initial (par cohorte, detenteurs precoces)
# Detenteur precoce = encours > 0 des le mois M+0 ou M+1 dans la famille.
TMAX = 60
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for ax, fam in zip(axes.ravel(), ENC_FAMS):
    early = set(panel[(panel["family"] == fam) & (panel["tenure"] <= 1) & (panel["encours"].fillna(0) > 0)]["identifiant_tiers"])
    sub = panel[(panel["family"] == fam) & (panel["identifiant_tiers"].isin(early)) &
                (panel["tenure"].between(0, TMAX)) & (panel["encours"].fillna(0) > 0)].copy()
    sub["cohort"] = sub["identifiant_tiers"].map(cohort_of)
    for coh in sorted(sub["cohort"].dropna().unique()):
        tr = sub[sub["cohort"] == coh].groupby("tenure")["encours"].mean()
        if len(tr) > 3: ax.plot(tr.index, tr.values, lw=1.5, label=int(coh))
    ax.set_title(f"{FAM_LABELS[fam]} — equipes des M+0/M+1"); ax.set_xlabel("Tenure"); ax.set_ylabel("Encours moyen (EUR)")
    ax.legend(fontsize=7, title="Cohorte")
fig.suptitle("Trajectoire d'encours depuis l'equipement initial, par cohorte", fontsize=15)
plt.tight_layout(); save_fig("s2_traj_initial")

## 3. Quel revenu les nouveaux clients genèrent-ils ?

PnB (NBI) genere sur les premiers mois et par annee de vie, sa concentration, et sa
repartition par produit. Ventilations selon les 6 caracteristiques.

In [ ]:
# 3.1 — PnB moyen genere a M+3 / M+6 / M+12 (NBI cumule) selon caracteristiques
PNB_HOR = [3, 6, 12]
pnb = {f"M+{X}": masked(cum_nbi_client(X), X) for X in PNB_HOR}
tbl = {f"PnB_median_M+{X}": round(masked(cum_nbi_client(X), X).median(), 0) for X in PNB_HOR}
print("PnB cumule median (tous clients eligibles) :", tbl)
break_by_multi(pnb, ylabel="PnB cumule moyen (EUR)",
               title="PnB cumule genere a M+3/M+6/M+12 par caracteristique", fig_name="s3_pnb_horizons")

In [ ]:
# 3.2 — PnB moyen genere chaque annee Y1 -> Y5 selon caracteristiques
YEARS_T = [1, 2, 3, 4, 5]
pnb_y = {f"Y{y}": masked(nbi_year(y), y * 12 - 1) for y in YEARS_T}
tbl = {f"Y{y}": round(masked(nbi_year(y), y * 12 - 1).mean(), 0) for y in YEARS_T}
print("PnB annuel moyen (par annee de vie) :", tbl)
break_by_multi(pnb_y, ylabel="PnB annuel moyen (EUR)",
               title="PnB genere par annee de vie (Y1->Y5) par caracteristique", fig_name="s3_pnb_years")

In [ ]:
# 3.3 — Concentration du PnB et des encours (Pareto / Lorenz)
def lorenz(ax, series, color, label):
    v = series[series > 0].sort_values()
    if len(v) == 0: return
    cum = np.cumsum(v.values) / v.sum() * 100
    xx = np.arange(1, len(v) + 1) / len(v) * 100
    ax.plot(xx, cum, lw=2.5, color=color, label=label)

client_nbi = flux.groupby("identifiant_tiers")["nbi"].sum()
client_enc = panel.groupby("identifiant_tiers")["encours"].sum()
fig, ax = plt.subplots(figsize=(9, 7))
lorenz(ax, client_nbi, PURPLE, "PnB"); lorenz(ax, client_enc, CYAN, "Encours")
ax.plot([0, 100], [0, 100], "--", color="gray", alpha=0.6, label="Egalite parfaite")
ax.set_xlabel("% des clients (croissant)"); ax.set_ylabel("% du total"); ax.set_title("Concentration PnB & encours"); ax.legend()
plt.tight_layout(); save_fig("s3_concentration")
vd = client_nbi[client_nbi > 0].sort_values(ascending=False)
print("Concentration du PnB :")
for pct in [1, 5, 10, 20, 50]:
    k = max(int(len(vd) * pct / 100), 1)
    print(f"  Top {pct:2d}% des clients -> {vd.head(k).sum() / vd.sum() * 100:.1f}% du PnB")

In [ ]:
# 3.4 — PnB moyen genere par type de produit
fam_nbi = pd.Series({fam: panel[panel["family"] == fam]["nbi"].sum() for fam in FAM_ORDER})
fam_nbi = fam_nbi[fam_nbi > 0]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar([FAM_LABELS[f] for f in fam_nbi.index], fam_nbi.values / 1e6,
            color=[FAM_COLORS[f] for f in fam_nbi.index], edgecolor="white")
axes[0].set_title("PnB total par produit"); axes[0].set_ylabel("M EUR"); axes[0].tick_params(axis="x", rotation=20)
axes[1].pie(fam_nbi.values, labels=[FAM_LABELS[f] for f in fam_nbi.index], autopct="%1.0f%%",
            colors=[FAM_COLORS[f] for f in fam_nbi.index], startangle=90)
axes[1].set_title("Part de chaque produit dans le PnB")
plt.tight_layout(); save_fig("s3_pnb_by_product")
print("Contribution au PnB :")
for f in fam_nbi.sort_values(ascending=False).index:
    print(f"  {FAM_LABELS[f]:<18}: {fam_nbi[f]/fam_nbi.sum()*100:5.1f}%")

## 4. Comment calibrer le modèle ?

Persistance de l'equipement, des encours et du PnB entre M+3, M+12 et Y5 ; marge implicite
par produit par annee (sur detenteurs) ; niveau des deciles d'encours par produit par annee.

In [ ]:
# 4.1 — Taux d'equipement : M+3 vs M+12 et M+12 vs Y5 (retention de l'equipement, %)
def hold_set(fam, t):
    return set(panel[(panel["family"] == fam) & (panel["tenure"] == t) &
                     ((panel["encours"].fillna(0) > 0) | (panel["nbi"].fillna(0) > 0))]["identifiant_tiers"])
rows = []
for fam in FAM_ORDER:
    h3, h12, h60 = hold_set(fam, 3), hold_set(fam, 12), hold_set(fam, 60)
    e12 = eligible_clients(13); e60 = eligible_clients(61)
    keep_3_12  = (h3 & e12); keep_12_60 = (h12 & e60)
    r = {"famille": FAM_LABELS[fam]}
    r["retention_M3->M12_%"]  = round(len(keep_3_12 & h12) / max(len(keep_3_12), 1) * 100, 1)
    r["retention_M12->Y5_%"] = round(len(keep_12_60 & h60) / max(len(keep_12_60), 1) * 100, 1)
    rows.append(r)
print("Retention de l'equipement par produit :"); print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# 4.2 & 4.3 — Correlation encours et PnB : M+3 vs M+12, M+12 vs Y5 (Spearman, par produit)
def corr_at(getter, fam, t1, t2):
    a = getter(fam, t1); b = getter(fam, t2)
    a = a[a > 0]
    common = list(set(a.index) & set(b.index) & eligible_clients(t2 + 1))
    if len(common) < 30: return np.nan, 0
    return round(stats.spearmanr(a.loc[common], b.loc[common])[0], 3), len(common)

enc_get = lambda fam, t: value_at_tenure("encours", fam, t)
rows = []
for fam in ENC_FAMS:
    r312, n1 = corr_at(enc_get, fam, 3, 12)
    r1260, n2 = corr_at(enc_get, fam, 12, 60)
    rows.append({"famille": FAM_LABELS[fam], "enc_rho_M3_M12": r312, "n1": n1,
                 "enc_rho_M12_Y5": r1260, "n2": n2})
print("Correlation ENCOURS (Spearman, meme produit) :"); print(pd.DataFrame(rows).to_string(index=False))

# PnB : correlation des NBI cumules (total client) M+3 vs M+12, M+12 vs Y5
def pnb_win(a, b):
    x = cum_nbi_client(b) - (cum_nbi_client(a) if a > 0 else 0)
    return x
def pnb_corr(a1, a2):
    A = cum_nbi_client(a1); B = cum_nbi_client(a2)
    common = list(set(A.index) & set(B.index) & eligible_clients(a2))
    return (round(stats.spearmanr(A.loc[common], B.loc[common])[0], 3), len(common)) if len(common) > 30 else (np.nan, 0)
r1 = pnb_corr(3, 12); r2 = pnb_corr(12, 60)
print(f"\nCorrelation PnB cumule : M+3 vs M+12 rho={r1[0]} (n={r1[1]}) | M+12 vs Y5 rho={r2[0]} (n={r2[1]})")

In [ ]:
# 4.4 — Marge implicite par produit PAR ANNEE de vie (NBI / encours, sur detenteurs)
# Corrige : numerateur et denominateur sur les MEMES lignes detentrices (encours > 0).
rows = []
for fam in ENC_FAMS:
    for y in [1, 2, 3, 4, 5]:
        lo, hi = (y - 1) * 12, y * 12
        sub = panel[(panel["family"] == fam) & (panel["tenure"] >= lo) & (panel["tenure"] < hi) &
                    (panel["encours"].fillna(0) > 0)]
        if len(sub) < 50: continue
        m = sub["nbi"].sum() / sub["encours"].sum()
        rows.append({"famille": FAM_LABELS[fam], "annee": f"Y{y}",
                     "marge_mensuelle_%": round(m * 100, 3), "marge_annualisee_%": round(m * 12 * 100, 2)})
marge_df = pd.DataFrame(rows)
print("Marge implicite par produit par annee (detenteurs) :")
print(marge_df.pivot(index="famille", columns="annee", values="marge_annualisee_%").to_string())

In [ ]:
# 4.5 — Niveau des deciles d'encours par produit PAR ANNEE de vie (detenteurs)
DECILES = [0.1, 0.25, 0.5, 0.75, 0.9]
for fam in ENC_FAMS:
    rows = []
    for y in [1, 2, 3, 4, 5]:
        lo, hi = (y - 1) * 12, y * 12
        v = panel[(panel["family"] == fam) & (panel["tenure"] >= lo) & (panel["tenure"] < hi) &
                  (panel["encours"].fillna(0) > 0)]["encours"]
        if len(v) < 50: continue
        rows.append({"annee": f"Y{y}", **{f"D{int(q*100)}": round(v.quantile(q), 0) for q in DECILES}})
    if rows:
        print(f"\n{FAM_LABELS[fam]} — quantiles d'encours (detenteurs) par annee :")
        print(pd.DataFrame(rows).to_string(index=False))